In [7]:
import json

f = open('Datasets/wikipedia.json')
data = json.load(f)
resultados = []
for elem in data:
    if len(elem["Texto"]) > 200:
        resultados.append(elem)

In [8]:
with open('Datasets/wikipedia.json', 'w') as fout:
    json.dump(resultados , fout, ensure_ascii=False, indent=2)

In [ ]:
import json

f = open('Datasets/wikipedia.json')
data = json.load(f)

In [ ]:
def obtener_respuesta(Titulo, Seccion, Texto):
    prompt=f"""Eres un chatbot encargado de extraer proposiciones atómicas de un texto. 
                Una proposición es atómica cuando es autocontenida: no necesita contexto u otra información para entenderse, por ejemplo: 'Morten Harket sufrió acoso escolar' es correcta mientras que 'Sufrió acoso escolar' no es autocontenida ya que no se sabe a quién hace referencia; es minimal: no se puede descomponer en proposiciones más sencillas; y es coherente: tiene sentido, es decir, es una frase completa con sujeto y predicado, una palabra suelta o una sigla no es una proposición. 
                Solo debes listar las proposiciones atómicas separadas por puntos y un salto de línea "\\n", sin dar ninguna información extra; por ejemplo: París es una ciudad. París es la capital de Francia.
                Dado el siguiente texto de ejemplo: 'Morten Harket nació en Kongsberg siendo el segundo de 5 hijos. Su padre, Reidar, fue médico en un hospital y su madre Henny fue maestra de economía del hogar en una escuela. Morten fue un chico soñador y sufrió acoso escolar en la escuela. Se destacaba en las clases de religión y pensó ingresar en un seminario para ser ministro en la Iglesia Luterana. Entró a formar parte de A-ha el 14 de septiembre de 1982, por invitación de Paul Waaktaar-Savoy y Magne Furuholmen. Sin embargo antes de esto fue el vocalista de un grupo local llamado Soldier Blue. A-ha fue un grupo de gran éxito en los años 80, siendo su canción «Take on Me» la de mayor éxito, de su disco debut «Hunting High and low» vendiendo millones de discos, donde Morten Hartet se convirtió rápidamente en un rompe corazones por aquellas primeras imágenes del vídeo musical «Take on Me» en donde su aspecto de galán nórdico causó gran sensación entre el público femenino. Después de que A-ha se separara en 1994 por conflictos internos, Morten emprendió una carrera como solista cantante y compositor y participó en proyectos medioambientales. Más importante para él fue la lucha por liberar Timor Oriental de la ocupación indonesia, un objetivo que fue finalmente cumplido en 1999. En 1996, Morten es el encargado de presentar junto a Ingvild Bryn el Festival de Eurovisión que se celebró en Oslo, donde cantó Heaven's Not For Saints como introducción a la gala (no como representante de Noruega). Morten ha sacado tres álbumes como solista: «Poetenes Evangelium» (1993) y «Vogts Villa» (1996) en noruego y «Wild Seed» (1995) en inglés, este último fue un gran éxito en Noruega, vendiendo más de 160,000 copias. Otros proyectos de Morten son: la banda sonora de la película «Kamilla og Tyven» (1987), en la que además interpretó un personaje, y «Coneheads» (1993). Otras bandas sonoras incluyen el tema central del musical «Sophie's World», «A Jester In Out Town» y la canción «Jungle Of Beliefs» del disco «Cultures Span The World». En el año 2007 presenta «Letter from Egypt» su regreso como solista y dando una pausa a su participación con el trío noruego A-ha. El 14 de noviembre de 2007 se difundió su primer sencillo «Movies» en la radio en Noruega, colocándose rápidamente en los primeros lugares. En diciembre de 2007 tuvo una presentación en el Premio Nobel de la Paz.'
                Serían proposiciones atómicas correctas: Morten Harket nació en Kongsberg. Morten Harket es el segundo de 5 hijos. El padre de Morten Harket se llama Reidar. Reidar Harket fue médico de un hospital.
                Serían proposiciones atómicas incorrectas: Él nació en Kongsberg. (No autocontenida); Su madre fue maestra de economía del hogar. (No autocontenida); Morten Harket fue un chico soñador y sufrió acoso escolar. (No minimal, se puede separar en: Morten Harket fue un chico soñador. Morten Harket sufrió acoso escolar.); Morten Harket fue astronauta. (Incorrecta, ya que esa infromación no está en el texto);
                Por último, es importante que si no logras extraer proposiciones de ninguna forma pongas: 'No se pueden extaer proposiciones.'."\n
                Utiliza el título y la sección para contextualizar las proposiciones en caso de que esa información no venga directamente en el texto.
                Dado el siguiente título: {Titulo}; la siguiente sección: {Seccion} y el siguiente texto: {Texto}\n
                Lista las proposiciones atómicas:
           """
    return prompt

In [ ]:
import ollama
import time
import pandas as pd

resultado = {'Instruccion': [], 'Respuesta': []}
cont = 0
stop = 10000
for elem in data:
    cont += 1
    print(f"Fase {cont} de {stop}; {round((cont/stop)*100, 2)} ", end='\r')
    if cont > stop:
        break
    print(f"Fase {cont} de {stop}", end='\r')
    instr = obtener_respuesta(elem["Título"], elem["Sección"], elem["Texto"])
    resultado['Respuesta'].append(ollama.generate(model='gpt-oss:latest', prompt=instr)["response"])
    resultado['Instruccion'].append(elem["Texto"])

    
df = pd.DataFrame(data=resultado)
df.to_csv('Datasets/ResultadoProposiciones_mejorado.csv', index=False)